# Lesson 25 Lab — Failure Modes: Outliers, Long Context, MoE, and Small Batches

**Puzzle:** Where should a quantized system be expected to fail first?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

Quantization often fails by regime rather than on average. Activation outliers amplify weight error, a shifted domain changes channel importance, tiny batches expose launch/dequant overhead, long context expands cache pressure, and MoE routing concentrates work unevenly. A failure matrix makes those reversals visible before production does.


## 0. Predict before running

1. Predict which synthetic case produces the largest W4 output RMSE.
2. Predict whether the reference W4 path is faster in every batch/distribution case.
3. Design separate tests for long-context cache and MoE routing, which this linear probe does not contain.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

Failure modes map to mechanisms: range outliers, distribution shift, long-context cache/attention, MoE routing imbalance, and small irregular GEMMs.

- Outliers enlarge scale and waste codes on ordinary values.
- Long context expands cache and can expose positional or attention regressions.
- MoE routing and small batches create irregular, overhead-sensitive shapes.


## 2. Derive the mechanism

One outlier can enlarge a group scale; shifted inputs change layer-output sensitivity; batch-one and routed experts reduce matrix sizes and make launch/dequant overhead visible.

For fixed weight error ΔW, output error is `XΔWᵀ`; scaling or shifting X directly changes its magnitude and direction. This explains why a quantizer calibrated on ordinary activations can degrade under outliers or domain shift without any weight bytes changing. Small batches add a systems failure mode because fixed launch, unpack, or scale overhead is amortized over less work.

Long context and MoE require additional objects: cache bytes/attention error and expert-routing load balance. They belong in the matrix but cannot be inferred from one dense linear layer.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "25-quantization-failure-modes"
device = require_cuda()
torch.manual_seed(2026 + 25)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | BF16 matrix multiplication in four controlled input regimes |
| Candidate | the same multiplication with group-128 INT4-dequantized weights |
| Held constant | weight matrix and quantizer; only batch/distribution regime changes |
| Measurements | output RMSE/cosine/max error and median/p90 timing per regime |
| Evidence | `pytorch-gpu` |

**Experiment:** Stress an INT4 linear reference with ordinary inputs, activation outliers, narrow batches, and shifted distributions on CUDA.


## 5. Read the experiment code

The lab holds weights fixed and stresses ordinary, outlier, shifted, and small-batch inputs, preserving each condition instead of averaging them together.

The notebook quantizes one weight matrix once, then evaluates ordinary, batch-1, activation-outlier, and shifted-domain inputs. Each row carries both numerical error and timing for baseline/candidate. That paired design prevents a quality failure from being hidden by a small speed result.

The candidate is a dequantized PyTorch reference tensor, not a packed production W4 kernel. Timing differences therefore illustrate regime sensitivity of the composed path, not an INT4 hardware speed claim.

Only after these variables match the protocol should the cell be executed.


In [2]:
w=torch.randn(1024,1024,device=device); _,_,dq=symmetric_quantize(w,bits=4,group_size=128); rows=[]
cases={"ordinary":torch.randn(32,1024,device=device),"small_batch":torch.randn(1,1024,device=device),
       "activation_outliers":torch.randn(32,1024,device=device),"shifted_domain":torch.randn(32,1024,device=device)*3+2}
cases["activation_outliers"][:,::73]*=30
for name,x in cases.items(): rows.append({"case":name,"output_error":error_metrics(x@w.t(),x@dq.t()),
    "bf16_timing":cuda_benchmark(lambda:x@w.t(),warmup=3,repeats=12),"reference_w4_timing":cuda_benchmark(lambda:x@dq.t(),warmup=3,repeats=12)})
result=base_result(25,"pytorch-gpu"); result.update({"failure_matrix":rows,
    "conclusion":"Condition-specific tests exposed reversals that an aggregate average could conceal."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Ordinary RMSE | 3.729115 |
| Small-batch RMSE | 3.694930 |
| Activation-outlier RMSE | 14.075421 |
| Shifted-domain RMSE | 13.752637 |
| Largest shifted max error | 67.176849 |


## 7. Interpret rather than merely print

Ordinary and batch-1 RMSE were about 3.73 and 3.69. Activation outliers raised RMSE to 14.0754 and max error to 59.1648; the shifted domain produced RMSE 13.7526 and max error 67.1768. Timing changes stayed tiny and varied by row.

An aggregate over all four cases could hide the roughly 3.7x error jump in the shifted regimes. The correct response is a targeted calibration, fallback, or rejection rule—not a global statement that W4 is acceptable.

**Inspection rule:** Keep a failure matrix by condition. Average error over mixed cases can conceal the exact reversal condition.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The measured tensors and operations ran on CUDA through PyTorch. The result does not name a separate production backend unless an operator trace identifies it.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "Condition-specific tests exposed reversals that an aggregate average could conceal.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:46:14+00:00",
  "failure_matrix": [
    {
      "bf16_timing": {
        "median_ms": 0.019584,
        "p90_ms": 0.020288,
        "repeats": 12,
        "samples_ms": [
          0.030048,
          0.022048,
          0.020288,
          0.01952,
          0.019744,
          0.019648,
          0.01888,
          0.019168,
          0.017984,
          0.019072,
          0.01968,
          0.019104
        ],
        "warmup": 3
      },
      "case": "ordinary",
      "output_error": {
        "cosine": 0.99317932,
        "mae": 2.97387171,
        "max_abs": 17.41661835,
        "rmse": 3.72911549
      },
 

## 9. Make the bounded decision

> Design negative tests from known mechanisms and preserve a fallback for the slice that fails.

**Acceptance/rollback:** Maintain a condition-by-metric failure matrix with reversal thresholds and reproduce each failure independently before assigning a fallback.

**Failure analysis:** One stress tensor cannot represent production tail frequency, and synthetic timing with dequantized weights is not a native backend result. A matrix that lists long context or MoE without actually constructing cache or routing evidence would also be misleading; unexecuted axes must remain marked as future gates.


## 10. Extend the evidence

Add a real KV-cache length sweep, rare-language/code/tool-use activation captures, and a toy MoE with expert-load imbalance. Define acceptance by slice, not only aggregate. Use the failing rows to design mixed-bit fallbacks and then rerun the full matrix.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
